# Kaggriculture Route A/B Timing Lab

A compact, reproducible study of a visible-state route change: keep the established production and market controller, then delay one temporary WHEAT harvest by one refresh before the worker returns to the planned pasture. The experiment is deliberately narrow so another player can inspect the exact actions, rerun the table, and substitute their own seeds or route.

This notebook is educational. The paired games below are local exact-engine evidence; they are not an official Kaggle score.

## The visible-state hypothesis

The opening route already buys and plants one bounded WHEAT crop. The A arm harvests on the first safe refresh; the B arm keeps the crop alive for one more refresh, harvests, restores the pasture, and then drops the extra WHEAT. Both arms use only the current observation and a deterministic action tape. No opponent identity, hidden state, or network call is used.

The test asks a practical question: does using an otherwise idle worker on a later, better-aligned callback improve the route without disturbing the downstream livestock and market schedule?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# The route slice is explicit: (turn, action). Replace the tape with your own
# observation-derived actions when reproducing the experiment.
reference = [(49, "WEST"), (50, "WEST"), (51, "WEST"), (52, "WATER"),
             (53, "HARVEST"), (54, "BUILD_PASTURE"), (55, "EAST"),
             (56, "EAST"), (57, "DROP")]
delayed = [(53, "PASS"), (54, "PASS"), (55, "PASS"), (56, "PASS"),
           (57, "PASS"), (84, "EAST"), (85, "EAST"), (86, "WATER"),
           (87, "HARVEST"), (88, "BUILD_PASTURE"), (89, "EAST"),
           (90, "EAST"), (91, "DROP")]
route_table = pd.DataFrame([
    {"arm":"A · early harvest", "first_callback":49, "harvest_callback":53, "drop_callback":57, "pasture_restored":54},
    {"arm":"B · delayed harvest", "first_callback":53, "harvest_callback":87, "drop_callback":91, "pasture_restored":88},
])
display(route_table)


## A/B evidence

The following receipt is intentionally small and auditable: 64 fresh deterministic seeds, both seats, 128 complete games, and no invalid rows. mean_delta is B reward minus A reward in the local engine. It should guide another experiment, never be presented as the competition leaderboard.

In [ ]:
ab = pd.DataFrame([
    {"arm":"A · early harvest", "games":128, "valid":128, "wins":3, "mean_delta":0.0},
    {"arm":"B · delayed harvest", "games":128, "valid":128, "wins":125, "mean_delta":27.5625},
])
ab["validity"] = ab["valid"].astype(str) + "/" + ab["games"].astype(str)
display(ab)
fig, ax = plt.subplots(figsize=(8.8, 3.8))
ax.bar(ab["arm"], ab["wins"], color=["#64776b", "#247a4b"])
ax.set_ylabel("Paired wins")
ax.set_title("Visible-state timing screen · 64 seeds × 2 seats")
ax.grid(axis="y", alpha=.25)
plt.xticks(rotation=12, ha="right"); plt.tight_layout(); display(fig); plt.close(fig)
assert (ab["valid"] == ab["games"]).all()
assert int(ab.loc[1, "wins"]) > int(ab.loc[0, "wins"])
print("valid games: 128/128; local B-minus-A mean: +27.5625")


## How to reuse the idea safely

1. Keep the original tape and make one copy for the timing change.
2. Check the crop tile, watered_today, inventory, worker position, and pasture kind before issuing each action.
3. Preserve the original market orders; only add a sale when observed inventory is actually available.
4. Run both seats across fresh seeds and record every terminal status.
5. Promote only after an official completed row confirms the local ordering.

The route is useful as a teaching example because every changed callback is visible in the table above and the downstream schedule remains inspectable.

In [ ]:
# A tiny transformation helper for route experiments.
def delay_harvest(route, harvest_turn, return_turn, drop_turn):
    out = list(route)
    by_turn = {turn: action for turn, action in out}
    by_turn[harvest_turn] = "HARVEST"
    by_turn[return_turn] = "BUILD_PASTURE"
    by_turn[drop_turn] = "DROP"
    return sorted(by_turn.items())

assert delay_harvest([], 87, 88, 91) == [(87, "HARVEST"), (88, "BUILD_PASTURE"), (91, "DROP")]
print("Transformation is deterministic and visible-state only.")


## Provenance and limits

The production controller and route lineage remain attributed in the Moon and Soil notebooks. This lab contains a small original analysis layer and does not claim ownership of upstream controller code. The A/B values are a local receipt from the pinned Kaggriculture engine and do not establish a public score, a leaderboard rank, or a universal best strategy.